# Pose Transformer training

Curated portfolio copy of the original coursework notebook. Run cells in order with the required data and dependencies.


## Environment


In [ ]:
!pip install boto3 -q
!pip install opencv-python torch numpy torchvision
!pip install mediapipe


## Dataset download


In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import os

# Connect to S3 without authentication (public bucket)
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

bucket_name = 'prism-mvta'
prefix = 'training-and-validation-data/'
download_dir = './video-data'

os.makedirs(download_dir, exist_ok=True)

# List all objects in the S3 path
paginator = s3.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

video_names = []

for page in pages:
    if 'Contents' not in page:
        print("No files found at the specified path! Go and complain to the TAs!")
        break

    for obj in page['Contents']:
        key = obj['Key']
        filename = os.path.basename(key)

        if not filename:
            continue

        video_names.append(filename)

        local_path = os.path.join(download_dir, filename)
        print(f"Downloading: {filename}")
        s3.download_file(bucket_name, key, local_path)

print("\n" + "="*50)
print("Downloaded videos:")
print("="*50)
for name in video_names:
    print(name)

print(f"\nTotal: {len(video_names)} files")

## Pose extraction and features


In [ ]:
# Here is a basic implementation of the above two TODOs. You can assume the first TODO is completed correctly.

# Please modify this code to suit you best, as you decide on your preferred model architecture.

# For example, below here we are padding every video to 1,000 frames. That may or may not be a good idea.
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
from pathlib import Path
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import requests
import math
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix


seed = 1
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)


def extract_and_save_poses(video_dir, output_dir):

    # Download mediapipe model
    model_path = 'pose_landmarker_full.task'
    url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task"
    if not os.path.exists(model_path):
        print(f"Downloading MediaPipe Model")
        with open(model_path, 'wb') as f: f.write(requests.get(url).content)

    # MediaPipe Setup
    BaseOptions = mp.tasks.BaseOptions
    PoseLandmarker = mp.tasks.vision.PoseLandmarker
    PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
    VisionRunningMode = mp.tasks.vision.RunningMode

    os.makedirs(output_dir, exist_ok=True)

    # Configure Landmarker for VIDEO mode
    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.VIDEO
    )

    with PoseLandmarker.create_from_options(options) as landmarker:
        video_files = list(Path(video_dir).glob('*.mp4'))

        for video_path in video_files:
            output_path = Path(output_dir) / f"{video_path.stem}.npz"
            if output_path.exists():
                print(f"Skip (already exists): {output_path.name}")
                continue

            print(f"Processing: {video_path.name}")
            cap = cv2.VideoCapture(str(video_path))
            fps = cap.get(cv2.CAP_PROP_FPS)

            all_frames_data = []
            frame_count = 0

            with vision.PoseLandmarker.create_from_options(options) as landmarker:
                while cap.isOpened():
                    ret, frame = cap.read()
                    if not ret: break

                    # RGB
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

                    # Calculate timestamp
                    timestamp_ms = int((frame_count * 1000) / fps)

                    # Detect
                    result = landmarker.detect_for_video(mp_image, timestamp_ms)

                    # 33 joints * (x, y, z, visibility)
                    frame_coords = np.zeros((33, 4))
                    if result.pose_landmarks:
                        for i, lm in enumerate(result.pose_landmarks[0]):
                            frame_coords[i] = [lm.x, lm.y, lm.z, lm.presence]

                    all_frames_data.append(frame_coords)
                    frame_count += 1

            cap.release()

            # Save as compressed npz
            np.savez_compressed(
                output_path,
                coords=np.array(all_frames_data, dtype=np.float32), #(T,33,4)
                fps=np.array(fps)
            )
            print(f"Saved {frame_count} frames")
    print('Extraction complete')

# Run extraction
extract_and_save_poses('./video-data', './mediapipe-skeleton')

In [ ]:
from sklearn.model_selection import train_test_split
from collections import Counter
import os


def balanced_split(data_dir, test_size=0.2, random_state=42):

    files = np.array(sorted([f for f in os.listdir(data_dir) if f.endswith('.npz')]))
    labels = np.array([int(f.split('_')[0]) for f in files])

    train_indices = []
    val_indices = []

    counts = Counter(labels)

    # Separate classes that can be stratified from those that are too small
    stratify_files = []
    stratify_labels = []

    for label in counts:
        label_indices = np.where(labels == label)[0]

        if counts[label] < 2:
            # FORCE to training
            train_indices.extend(label_indices)
        else:
            for idx in label_indices:
                stratify_files.append(files[idx])
                stratify_labels.append(labels[idx])

    # 3. Stratified split for the remaining data
    if stratify_files:
        x_train, x_val, y_train, y_val = train_test_split(
            stratify_files,
            stratify_labels,
            test_size=test_size,
            stratify=stratify_labels,
            random_state=random_state
        )

    return list(x_train) + [files[i] for i in train_indices], list(x_val)

# Run split
train_files, val_files = balanced_split('mediapipe-skeleton', test_size=0.2)
print(f"Train: {len(train_files)} | Val: {len(val_files)}")

# Verify distribution
train_labels = [int(f.split('_')[0]) for f in train_files]
print("Train distribution:", Counter(train_labels))

In [ ]:
# helper functions

def sample_clip_bucket_jitter(landmarks: np.ndarray, L: int = 64, seed: int | None = None):
    assert landmarks.ndim == 3 and landmarks.shape[1:] == (33, 4)

    T = landmarks.shape[0]
    rng = np.random.default_rng(seed)

    if T == 0:
        return np.zeros((L, 33, 4), dtype=np.float32)

    # Split timeline into L bins
    edges = np.linspace(0, T, L + 1)

    idx = np.empty(L, dtype=np.int64)
    for i in range(L):
        lo = int(edges[i])
        hi = int(edges[i + 1]) - 1
        lo = max(0, min(lo, T - 1))
        hi = max(lo, min(hi, T - 1))

        if hi < lo:
            hi = lo  # degenerate bin

        idx[i] = rng.integers(lo, hi + 1)

    idx.sort()  # preserve temporal order
    return landmarks[idx].astype(np.float32)

def synthesize_stitched_clip(landmarks_a: np.ndarray, label_a: int, landmarks_b: np.ndarray, label_b: int,
                             L: int = 64, transition_frames: int = 5, seed: int | None = None):
    """
    Stitch two clips together.
    1. Concatenates landmarks_a and landmarks_b
    2. Applies linear interpolation at the seam
    3. Samples the combined trajectory into L frames
    """
    # 1. Calculate new label
    new_label = label_a + label_b

    # 2. Smooth the transition at the seam
    last_frame_a = landmarks_a[-1:]   # (1, 33, 4)
    first_frame_b = landmarks_b[0:1]  # (1, 33, 4)

    # Create transition frames (Lerp)
    alphas = np.linspace(0, 1, transition_frames).reshape(-1, 1, 1)
    seam_transition = (1 - alphas) * last_frame_a + alphas * first_frame_b

    # Combine
    combined_raw = np.concatenate([
        landmarks_a[:-1],
        seam_transition,
        landmarks_b[1:]
    ], axis=0)

    # 3. Random sample to get exactly L frames
    synthesized_landmarks = sample_clip_bucket_jitter(combined_raw, L=L, seed=seed)

    return synthesized_landmarks.astype(np.float32), new_label

def save_clip(output_dir, label,clip,sample_id,method,source):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    out_path = output_dir / f"{label}__{method}__{sample_id:05d}.npz"
    np.savez_compressed(
        out_path,
        clip=clip.astype(np.float32),
        label=int(label),
        method=method,
        source=source,
    )

In [ ]:
import random
from pathlib import Path

def build_trainset(input_dir, train_files, output_dir, target_samples_per_label=50, L=64):
    """
    2. Uses jittered sampling for existing labels.
    3. Stitches clips to create data for missing labels.
    """
    label_buckets = {}
    for f in train_files:
        label = int(f.split('_')[0])
        data = np.load(os.path.join(input_dir, f))
        label_buckets.setdefault(label, []).append(data['coords'])

    existing_labels = sorted(label_buckets.keys())
    max_label = 10
    sample_id = 0

    # Iterate through every possible label from 1 to max_label
    for target_label in range(1, max_label + 1):
        generated = 0

        # CASE 1: Label exists - Jitter Sampling
        if target_label in label_buckets:
            source_trajs = label_buckets[target_label]

            # 1) Ensure every original video has at least ONE clip
            for i, base_traj in enumerate(source_trajs):
                clip = sample_clip_bucket_jitter(base_traj, L=L)
                save_clip(
                    output_dir=output_dir,
                    label=target_label,
                    clip=clip,
                    sample_id=sample_id,
                    method="orig",
                    source=f"video_{i}",
                )
                sample_id += 1
                generated += 1

            generated = len(source_trajs)

            # 2) Top up with additional random samples
            while generated < target_samples_per_label:
                base_traj = random.choice(source_trajs)
                clip = sample_clip_bucket_jitter(base_traj, L=L)
                save_clip(
                    output_dir=output_dir,
                    label=target_label,
                    clip=clip,
                    sample_id=sample_id,
                    method="jitter",
                    source="random",
                )
                sample_id += 1
                generated += 1


        # CASE 2: Label is missing - Stitching
        else:
            while generated < target_samples_per_label:
                for a in existing_labels:
                    b = target_label - a
                    if b in label_buckets:
                        traj_a = random.choice(label_buckets[a])
                        traj_b = random.choice(label_buckets[b])

                        clip, _ = synthesize_stitched_clip(
                            traj_a, a, traj_b, b, L=L
                        )

                        save_clip(
                            output_dir=output_dir,
                            label=target_label,
                            clip=clip,
                            sample_id=sample_id,
                            method="stitch",
                            source=f"{a}+{b}",
                        )
                        sample_id += 1
                        generated += 1
                        break

    print("Balanced dataset saved to:", output_dir)

build_trainset('mediapipe-skeleton', train_files, "expanded/train")

In [ ]:
def build_valset(
    input_dir: str,
    val_files: list[str],
    output_dir: str,
    L: int = 64,
):
    """
    Build validation set by UNIFORM sampling L frames per video.
    """
    input_dir = str(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    saved = 0

    for f in val_files:
        label = int(f.split("_")[0])
        data = np.load(os.path.join(input_dir, f))
        coords = data["coords"].astype(np.float32)  # (T,33,4)
        T = coords.shape[0]

        # Uniform deterministic sampling with padding
        if T <= 0:
            clip = np.zeros((L, 33, 4), dtype=np.float32)
            idx = np.zeros((L,), dtype=np.int64)
        elif T >= L:
            idx = np.linspace(0, T - 1, L, dtype=np.int64)
            clip = coords[idx]
        else:
            # T < L: take all frames then pad by repeating last
            pad = np.repeat(coords[-1][None, :, :], repeats=(L - T), axis=0)
            clip = np.concatenate([coords, pad], axis=0)
            idx = np.concatenate([np.arange(T, dtype=np.int64), np.full(L - T, T - 1, dtype=np.int64)])

        out_path = output_dir / f"{label}__val__{Path(f).stem}.npz"
        np.savez_compressed(
            out_path,
            clip=clip.astype(np.float32),   # (L,33,4)
            label=int(label),
            method="uniform",
            source=Path(f).stem,
            idx=idx,
        )
        saved += 1

    print(f"Saved {saved} validation clips to: {output_dir}")

build_valset(
    input_dir="mediapipe-skeleton",
    val_files=val_files,
    output_dir="expanded/val",
)

In [ ]:
import torch
from torch.utils.data import Dataset

class SkeletonDataset(Dataset):
    def __init__(self, data_dir, is_train=True, L=64):
        self.data_dir = data_dir
        self.file_list = [f for f in os.listdir(data_dir) if f.endswith('.npz')]
        self.is_train = is_train
        self.L = L

        # Indices in MediaPipe:
        # Shoulders (11,12), Elbows (13,14), Wrists (15,16), Hips (23,24)
        self.selected_joints = [11, 12, 13, 14, 15, 16, 23, 24]

    def __len__(self):
        return len(self.file_list)

    def _apply_augmentation(self, traj):

        # 1. Random Horizontal Flip (Mirroring)
        if np.random.rand() > 0.5:
            traj[:, :, 0] = 1.0 - traj[:, :, 0]
            left = [11, 13, 15, 23]
            right = [12, 14, 16, 24]
            traj[:, left + right] = traj[:, right + left]

        # 2. Random Rotation
        angle = np.radians(np.random.uniform(-10, 10))
        cos_a, sin_a = np.cos(angle), np.sin(angle)
        x, y = traj[:, :, 0].copy(), traj[:, :, 1].copy()
        traj[:, :, 0] = x * cos_a - y * sin_a
        traj[:, :, 1] = x * sin_a + y * cos_a

        # 3. Random Scaling (Zoom in/out)
        scale = np.random.uniform(0.9, 1.1)
        traj[:, :, :3] *= scale

        return traj

    def _calculate_angle(self, a, b, c):
        """Calculates the angle at joint 'b' given points a, b, and c."""
        ba = a - b
        bc = c - b
        norm_ba = np.linalg.norm(ba, axis=-1, keepdims=True)
        norm_bc = np.linalg.norm(bc, axis=-1, keepdims=True)

        cosine_angle = np.sum(ba * bc, axis=-1, keepdims=True) / (norm_ba * norm_bc + 1e-6)
        angle = np.arccos(np.clip(cosine_angle, -1.0, 1.0))
        return angle

    def _robust_center(self, feats):
        """
        Centering Logic: Hips -> Shoulders -> Global Mean
        """
        hip_vis = (feats[:, 6, 3] + feats[:, 7, 3]) / 2.0
        shld_vis = (feats[:, 0, 3] + feats[:, 1, 3]) / 2.0

        if np.mean(hip_vis) > 0.5:
            center = (feats[:, 6, :3] + feats[:, 7, :3]) / 2.0
        elif np.mean(shld_vis) > 0.5:
            center = (feats[:, 0, :3] + feats[:, 1, :3]) / 2.0
        else:
            center = np.mean(feats[:, :, :3], axis=1)

        feats[:, :, :3] -= center[:, np.newaxis, :]
        return feats

    def _engineer_features(self, traj):
        """
        1. Centering
        2. Angle Calculation (Elbows & Shoulders)
        3. Velocity calculation
        """
        # Slice joints
        # Map: 0:L_sh, 1:R_sh, 2:L_el, 3:R_el, 4:L_wr, 5:R_wr, 6:L_hip, 7:R_hip
        feats = traj[:, self.selected_joints, :].copy()

        # 1. Apply Robust Centering
        feats = self._robust_center(feats)

        # 2. Compute Angles
        # Left Elbow (Shoulder-Elbow-Wrist)
        l_elbow_angle = self._calculate_angle(feats[:,0,:3], feats[:,2,:3], feats[:,4,:3])
        # Right Elbow
        r_elbow_angle = self._calculate_angle(feats[:,1,:3], feats[:,3,:3], feats[:,5,:3])
        # Left Shoulder (Hip-Shoulder-Elbow)
        l_shld_angle = self._calculate_angle(feats[:,6,:3], feats[:,0,:3], feats[:,2,:3])
        # Right Shoulder
        r_shld_angle = self._calculate_angle(feats[:,7,:3], feats[:,1,:3], feats[:,3,:3])

        # 3. Assemble Feature Vector per frame
        # Positions (24) + Visibilities (8) + Angles (4) = 36 features
        pos_vis = feats.reshape(self.L, -1) # Flatten x,y,z,v
        angles = np.concatenate([l_elbow_angle, r_elbow_angle, l_shld_angle, r_shld_angle], axis=-1)

        base_features = np.concatenate([pos_vis, angles], axis=-1) # (L, 36)

        # 4. Velocity Features
        velocity = np.zeros_like(base_features)
        velocity[1:] = base_features[1:] - base_features[:-1]

        # Final Vector
        return np.concatenate([base_features, velocity], axis=-1)

    def __getitem__(self, idx):
        data = np.load(os.path.join(self.data_dir, self.file_list[idx]))
        traj = data['clip'].astype(np.float32)
        label = int(self.file_list[idx].split('_')[0])

        if self.is_train:
            traj = self._apply_augmentation(traj)

        features = self._engineer_features(traj) # (64, 72)
        return torch.tensor(features), torch.tensor(label, dtype=torch.long)

In [ ]:
def get_dataloaders(
    train_dir: str,
    val_dir: str,
    batch_size: int = 32,
    num_workers: int = 0,
    pin_memory: bool = True,
):
    train_ds = SkeletonDataset(train_dir)
    val_ds = SkeletonDataset(val_dir)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False,
    )

    return train_loader, val_loader


## Model


In [ ]:
import torch
import torch.nn as nn
from huggingface_hub import HfApi, hf_hub_download

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=64, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Register as buffer
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x: [Batch, Sequence_Len, Feature_Dim]
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class PushUpTransformer(nn.Module):
    def __init__(self, input_dim=72, num_classes=10, d_model=128, nhead=4, num_layers=3, dropout=0.2):
        super().__init__()

        # 1. Feature Projection & Normalization
        self.embedding = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU()
        )

        # 2. Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, max_len=64, dropout=dropout)

        # 3. Transformer Encoder Body
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=512,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 4. Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # Input x shape: (Batch_Size, 64, 72)

        # Embed and Add Position
        x = self.embedding(x)      # -> (Batch, 64, 128)
        x = self.pos_encoder(x)    # -> (Batch, 64, 128)

        # Transformer Pass (Self-Attention)
        x = self.transformer(x)    # -> (Batch, 64, 128)

        # Global Average Pooling
        x = x.mean(dim=1)          # -> (Batch, 128)

        # Classify
        logits = self.classifier(x) # -> (Batch, num_classes)
        return logits



## Training


In [ ]:
import time
import torch.optim as optim
import psutil
from sklearn.metrics import confusion_matrix

def get_system_ram():
    """Returns the current System RAM usage in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)

def calculate_metrics(logits, y):
    pred_counts = logits.argmax(dim=1) + 1
    acc = (pred_counts == y).float().mean().item()
    off_by_1 = (torch.abs(pred_counts - y) <= 1).float().mean().item()
    mae = (pred_counts - y).abs().float().mean().item()
    return acc, off_by_1, mae

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    loss_sum, acc_sum, off1_sum, mae_sum, n = 0.0, 0.0, 0.0, 0.0, 0
    start = time.time()

    for x, y in loader:
        x, y = x.to(device), y.to(device).long()
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y - 1)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        bsz = y.size(0)
        loss_sum += loss.item() * bsz
        acc, off1, mae = calculate_metrics(logits, y)
        acc_sum += acc * bsz
        off1_sum += off1 * bsz
        mae_sum += mae * bsz
        n += bsz

    return loss_sum / n, acc_sum / n, off1_sum / n, mae_sum / n, time.time() - start

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum, acc_sum, off1_sum, mae_sum, n = 0.0, 0.0, 0.0, 0.0, 0
    all_preds, all_targets = [], []
    start = time.time()

    for x, y in loader:
        x, y = x.to(device), y.to(device).long()
        logits = model(x)
        loss = criterion(logits, y - 1)
        bsz = y.size(0)
        loss_sum += loss.item() * bsz
        acc, off1, mae = calculate_metrics(logits, y)
        acc_sum += acc * bsz
        off1_sum += off1 * bsz
        mae_sum += mae * bsz
        n += bsz
        pred_counts = (logits.argmax(dim=1) + 1).cpu().numpy()
        all_preds.extend(pred_counts)
        all_targets.extend(y.cpu().numpy())

    return loss_sum / n, acc_sum / n, off1_sum / n, mae_sum / n, all_preds, all_targets, time.time() - start

def train_model(train_loader, val_loader, lr=1e-4, epochs=200, patience=50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # --- Initialize Memory Tracking ---
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()

    initial_ram = get_system_ram()

    model = PushUpTransformer(num_classes=12).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    history = {
        "train_loss": [], "train_acc": [], "train_off1": [], "train_mae": [],
        "val_loss": [], "val_acc": [], "val_off1": [], "val_mae": [],
        "best_cm": None,
        "best_epoch": None,
        "peak_sys_ram_mb": 0,
        "peak_gpu_vram_mb": 0
    }

    best_val_mae = float('inf')
    best_val_acc = 0.0
    no_improve = 0
    total_start = time.time()

    for epoch in range(1, epochs + 1):
        tl, ta, to1, tm, t_time = train_epoch(model, train_loader, optimizer, criterion, device)
        vl, va, vo1, vm, v_preds, v_targets, v_time = evaluate(model, val_loader, criterion, device)
        scheduler.step(vl)

        # --- Track Peak Memory per Epoch ---
        current_ram = get_system_ram()
        if current_ram > history["peak_sys_ram_mb"]:
            history["peak_sys_ram_mb"] = current_ram

        if device.type == 'cuda':
            peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 2)
            if peak_vram > history["peak_gpu_vram_mb"]:
                history["peak_gpu_vram_mb"] = peak_vram

        history["train_loss"].append(tl)
        history["train_acc"].append(ta)
        history["train_off1"].append(to1)
        history["train_mae"].append(tm)
        history["val_loss"].append(vl)
        history["val_acc"].append(va)
        history["val_off1"].append(vo1)
        history["val_mae"].append(vm)

        print(f"Ep {epoch:02d} | "
              f"Loss {tl:.3f}/{vl:.3f} | "
              f"Acc {ta:.1%}/{va:.1%} | "
              f"Off-1 {to1:.1%}/{vo1:.1%} | "
              f"MAE {tm:.2f}/{vm:.2f} | "
              f"RAM {current_ram:.0f}MB")

        if (vm < best_val_mae) or (vm == best_val_mae and va > best_val_acc):
            best_val_mae = vm
            best_val_acc = va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

            # Store the actual vectors so we can map missing labels correctly
            history["best_epoch"] = epoch
            history["best_y_true"] = v_targets
            history["best_y_pred"] = v_preds
            history["best_cm"] = confusion_matrix(v_targets, v_preds)

            no_improve = 0
            print("  <-- Model Updated")
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping.")
                break

    # --- Final Resource Summary ---
    print("\n" + "="*30)
    print(f"FINAL RESOURCE REPORT")
    print(f"Peak System RAM: {history['peak_sys_ram_mb']:.2f} MB")
    if device.type == 'cuda':
        print(f"Peak GPU VRAM:   {history['peak_gpu_vram_mb']:.2f} MB")
    print(f"Total Time:      {(time.time() - total_start) / 60:.2f} min")
    print("="*30)

    if best_state:
        model.load_state_dict(best_state)
        torch.save(model.state_dict(), 'model.pt')

    return model, history

In [ ]:
# setting a manual seed for reproducibility
train_loader, val_loader = get_dataloaders('expanded/train', 'expanded/val')
torch.manual_seed(1)
model, history = train_model(train_loader, val_loader)

## Evaluation and plots


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix

def plot_training_diagnostics(history, val_loader, save_path):
    """
    Generates a professional 5-panel dashboard.
    Dynamically maps confusion matrix labels to the actual classes present in the data.
    """
    plt.style.use('seaborn-v0_8-muted')
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    epochs = range(1, len(history["train_loss"]) + 1)

    # --- Performance Plots (Loss, Acc, Off-1, MAE) ---
    axes[0].plot(epochs, history["train_loss"], label='Train Loss')
    axes[0].plot(epochs, history["val_loss"], label='Val Loss')
    axes[0].set_title("A. Loss", fontweight='bold')
    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], label='Train Acc', color='green')
    axes[1].plot(epochs, history["val_acc"], label='Val Acc', color='darkgreen', linestyle='--')
    axes[1].set_title("B. Exact Accuracy", fontweight='bold')
    axes[1].legend()

    axes[2].plot(epochs, history["train_off1"], label='Train Off-1', color='orange')
    axes[2].plot(epochs, history["val_off1"], label='Val Off-1', color='darkorange', linestyle='--')
    axes[2].set_title("C. Off-by-1 Accuracy (±1)", fontweight='bold')
    axes[2].legend()

    axes[3].plot(epochs, history["train_mae"], label='Train MAE', color='red')
    axes[3].plot(epochs, history["val_mae"], label='Val MAE', color='darkred', linestyle='--')
    axes[3].set_title("D. Mean Absolute Error", fontweight='bold')
    axes[3].legend()

    # --- E. Confusion Matrix ---
    if "best_y_true" in history and "best_y_pred" in history:
        y_true = np.array(history["best_y_true"])
        y_pred = np.array(history["best_y_pred"])

        # Identify all unique labels present in the actual data
        unique_labels = np.unique(np.concatenate([y_true, y_pred]))
        unique_labels.sort()

        # Re-calculate CM specifically for these labels
        cm = confusion_matrix(y_true, y_pred, labels=unique_labels)

        # Normalize for recall
        cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-9)

        sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", ax=axes[4],
                    xticklabels=unique_labels.astype(int),
                    yticklabels=unique_labels.astype(int), cbar=False)

        axes[4].set_title("E. Confusion Matrix (Actual Classes)", fontweight='bold')
        axes[4].set_xlabel("Predicted Push-up Count")
        axes[4].set_ylabel("True Push-up Count")
    else:
        axes[4].text(0.5, 0.5, "No CM Data stored.\nUpdate evaluate() to save\nbest_y_true and best_y_pred.",
                     ha='center', va='center')

    # --- F. Summary Statistics ---
    axes[5].axis('off')

    best_ep = int(np.argmin(history["val_mae"]) + 1)

    rows = [
        ("Best Epoch", f"{best_ep}"),
        ("Min Val MAE", f"{min(history['val_mae']):.3f}"),
        ("Max Val Acc", f"{max(history['val_acc']):.1%}"),
        ("Max Off-1 Acc", f"{max(history['val_off1']):.1%}"),
    ]
    axes[5].set_title('F. Final Summary', fontweight='bold')


    cell_text = [[k, v] for k, v in rows]
    table = axes[5].table(
        cellText=cell_text,
        colLabels=["Metric", "Value"],
        cellLoc="left",
        colLoc="left",
        loc="center",
        bbox=[0.08, 0.20, 0.84, 0.60],
    )

    table.auto_set_font_size(False)
    table.set_fontsize(14)

    for (r, c), cell in table.get_celld().items():
        cell.set_linewidth(1.5)
        cell.set_edgecolor("peru")
        if r == 0:
            cell.set_text_props(weight="bold")
            cell.set_facecolor((0.96, 0.90, 0.78, 0.65))
        else:
            if r % 2 == 1:
                cell.set_facecolor((1.0, 0.98, 0.94, 0.55))
            else:
                cell.set_facecolor((1.0, 1.0, 1.0, 0.0))

    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.show()

# plot_training_diagnostics(history, val_loader)

be = history["best_epoch"]

for k, v in history.items():
    if isinstance(v, list):
        history[k] = v[:be]
plot_training_diagnostics(history, val_loader, 'model_diagnostics.png')



In [ ]:
from sklearn.decomposition import PCA

def capture_activations(model, x, device=None):

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    model.eval().to(device)
    x = x.to(device)

    acts = {}
    hooks = []

    def save(name):
        def hook(module, inp, out):
            acts[name] = out.detach().cpu()
        return hook

    hooks.append(model.embedding.register_forward_hook(save("embedding")))
    hooks.append(model.pos_encoder.register_forward_hook(save("pos_encoder")))

    # each encoder layer
    for i, layer in enumerate(model.transformer.layers):
        hooks.append(layer.register_forward_hook(save(f"encoder_layer_{i}")))

    hooks.append(model.classifier[0].register_forward_hook(save("clf_linear1_out")))
    hooks.append(model.classifier[-1].register_forward_hook(save("logits")))

    with torch.no_grad():
        logits = model(x).detach().cpu()

    for h in hooks:
        h.remove()

    return logits, acts


def collect_pooled_reprs(model, dataloader, layer_name="encoder_layer_2", device=None, max_batches=None):
    X_list, y_list = [], []
    for bi, (x, y) in enumerate(dataloader):
        if max_batches is not None and bi >= max_batches:
            break
        _, acts = capture_activations(model, x, device=device)

        A = acts[layer_name]       # (B, 64, 128)
        pooled = A.mean(dim=1)     # (B, 128)

        X_list.append(pooled.numpy())
        y_list.append(y.detach().cpu().numpy())

    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    return X, y

def plot_pca_scatter(X, y, title="PCA of pooled representations", savepath=None):
    Z = PCA(n_components=2).fit_transform(X)

    plt.figure(figsize=(6, 5))
    sc = plt.scatter(Z[:, 0], Z[:, 1], c=y, s=15)  # class-coded
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title(title)
    plt.colorbar(sc, label="Class")
    plt.tight_layout()

    if savepath:
        plt.savefig(savepath, dpi=600)
    plt.show()


X0, y = collect_pooled_reprs(model, train_loader, layer_name="encoder_layer_0")
plot_pca_scatter(X0, y, title="PCA (Layer 0 pooled)", savepath="pca_layer0.png")

X2, y = collect_pooled_reprs(model, train_loader, layer_name="encoder_layer_2")
plot_pca_scatter(X2, y, title="PCA (Layer 2 pooled)", savepath="pca_layer2.png")


